# Credit Card Payment Protection Insurance: Behavioral Time Series Analysis
## American Express Default Prediction - Insurance Focus

**Author:** Khushi  
**Course:** Data Science Applications in Banking and Insurance  
**Date:** December 2024

---

## Table of Contents
1. [Setup & Data Download](#section1)
2. [Data Loading & Sampling](#section2)
3. [Exploratory Data Analysis](#section3)
4. [Feature Engineering](#section4)
5. [Model Training](#section5)
6. [Insurance Metrics](#section6)
7. [Early Intervention Analysis](#section7)
8. [Visualizations](#section8)
9. [Conclusions](#section9)

---
<a id='section1'></a>
## 1. Setup & Data Download

### 1.1 Install Required Packages

In [3]:
# Install packages (run once)
!pip install kaggle pandas numpy scikit-learn matplotlib seaborn --quiet

### 1.2 Import Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Set display options
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


### 1.3 Setup Kaggle API Credentials

**BEFORE RUNNING THIS CELL:**
1. Go to https://www.kaggle.com/competitions/amex-default-prediction
2. Sign in and click "Join Competition" → Accept rules
3. Go to Account Settings → API → Create New Token
4. Download `kaggle.json` and upload it to this notebook environment

In [5]:
# Setup Kaggle credentials
!mkdir -p ~/.kaggle

# OPTION A: If you uploaded kaggle.json to current directory
!cp kaggle.json ~/.kaggle/kaggle.json

# OPTION B: If kaggle.json is in Downloads (uncomment if needed)
 !cp ~/Downloads/kaggle.json ~/.kaggle/kaggle.json

!chmod 600 ~/.kaggle/kaggle.json

# Verify
!ls -la ~/.kaggle/

print("\n✓ Kaggle credentials configured!")

IndentationError: unexpected indent (3879662055.py, line 8)

### 1.4 Download Dataset

**Choose ONE option below:**
- **Option A:** Full competition data (~3.5GB, 15-20 min download)
- **Option B:** Use synthetic sample data (instant, for demonstration)

In [ ]:
# OPTION A: Download real data (RECOMMENDED)
# Uncomment the lines below to download

# !mkdir -p ~/amex_data
# %cd ~/amex_data
# !kaggle competitions download -c amex-default-prediction
# !unzip -q amex-default-prediction.zip
# print("✓ Data downloaded and extracted!")
# %cd ~

In [ ]:
# OPTION B: Create synthetic sample data (for quick testing)
# Comment this out if you downloaded real data above

print("Creating synthetic sample data for demonstration...")

np.random.seed(42)
n_customers = 5000
n_months = 13

data = []
for customer_id in range(n_customers):
    default = np.random.choice([0, 1], p=[0.74, 0.26])
    
    for month in range(n_months):
        # Simulate behavioral deterioration for defaulters
        if default == 1 and month > 6:
            payment_trend = -0.05 * (month - 6)
            util_increase = 0.08 * (month - 6)
        else:
            payment_trend = np.random.uniform(-0.02, 0.02)
            util_increase = np.random.uniform(-0.02, 0.02)
        
        total_months = month + 3
        year = 2017 if total_months <= 12 else 2018
        month_num = total_months if total_months <= 12 else total_months - 12
        
        data.append({
            'customer_ID': f'cust_{customer_id}',
            'S_2': f'{year}-{month_num:02d}-01',
            'P_2': 100 + payment_trend * 100 + np.random.normal(0, 10),
            'B_1': max(0, 0.3 + util_increase + np.random.normal(0, 0.1)),
            'D_39': np.random.randint(0, 10),
            'S_3': max(0, 50 + np.random.normal(0, 20)),
            'target': default
        })

df_raw = pd.DataFrame(data)
df_raw['S_2'] = pd.to_datetime(df_raw['S_2'])

print(f"✓ Created synthetic dataset: {len(df_raw):,} rows, {df_raw['customer_ID'].nunique():,} customers")
print(f"✓ Default rate: {df_raw.groupby('customer_ID')['target'].first().mean():.1%}")

---
<a id='section2'></a>
## 2. Data Loading & Sampling

### 2.1 Load Data

In [ ]:
# IF YOU DOWNLOADED REAL DATA - Load from files
# Uncomment and run:

# print("Loading real competition data...")
# df_train = pd.read_csv('~/amex_data/train_data.csv')
# df_labels = pd.read_csv('~/amex_data/train_labels.csv')
# df_raw = df_train.merge(df_labels, on='customer_ID', how='left')
# df_raw['S_2'] = pd.to_datetime(df_raw['S_2'])
# print(f"✓ Loaded {len(df_raw):,} rows")

# IF YOU USED SYNTHETIC DATA - It's already loaded as df_raw
print(f"Dataset ready: {df_raw.shape}")

### 2.2 Sample 5,000 Customers

To make computation faster, we sample 5,000 customers while maintaining all their time series data.

In [ ]:
# Sample customers
N_SAMPLE = 5000

all_customers = df_raw['customer_ID'].unique()
print(f"Total customers available: {len(all_customers):,}")

if len(all_customers) > N_SAMPLE:
    np.random.seed(42)
    sampled_customers = np.random.choice(all_customers, size=N_SAMPLE, replace=False)
    df_sample = df_raw[df_raw['customer_ID'].isin(sampled_customers)].copy()
    print(f"✓ Sampled {N_SAMPLE:,} customers")
else:
    df_sample = df_raw.copy()
    print(f"✓ Using all {len(all_customers):,} customers")

print(f"Sample dataset shape: {df_sample.shape}")
print(f"Date range: {df_sample['S_2'].min()} to {df_sample['S_2'].max()}")

---
<a id='section3'></a>
## 3. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Dataset Overview:")
print(f"  Unique customers: {df_sample['customer_ID'].nunique():,}")
print(f"  Total records: {len(df_sample):,}")
print(f"  Records per customer: {len(df_sample) / df_sample['customer_ID'].nunique():.1f}")

if 'target' in df_sample.columns:
    default_rate = df_sample.groupby('customer_ID')['target'].first().mean()
    print(f"  Default rate: {default_rate:.1%}")

print("\nFirst few rows:")
df_sample.head()

In [ ]:
# Column types
print("Column Summary:")
print(df_sample.dtypes.value_counts())

print("\nNumeric columns:")
numeric_cols = df_sample.select_dtypes(include=[np.number]).columns.tolist()
print(f"Total: {len(numeric_cols)}")
print(numeric_cols[:10], "...")

---
<a id='section4'></a>
## 4. Feature Engineering - Time Series Features

Creating behavioral trend features that capture payment deterioration patterns.

In [ ]:
print("Creating time series features...\n")

# Sort by customer and date
df_sample = df_sample.sort_values(['customer_ID', 'S_2'])

# Identify feature columns by prefix
payment_cols = [col for col in df_sample.columns if col.startswith('P_')]
balance_cols = [col for col in df_sample.columns if col.startswith('B_')]
spend_cols = [col for col in df_sample.columns if col.startswith('S_') and col != 'S_2']
delinq_cols = [col for col in df_sample.columns if col.startswith('D_')]

print(f"Found feature categories:")
print(f"  Payment (P_): {len(payment_cols)}")
print(f"  Balance (B_): {len(balance_cols)}")
print(f"  Spend (S_): {len(spend_cols)}")
print(f"  Delinquency (D_): {len(delinq_cols)}")

In [ ]:
# Create time-based features
print("\nEngineering time series features...")

feature_count = 0

# Payment trends (first 3 payment columns)
for col in payment_cols[:3]:
    if col in df_sample.columns:
        df_sample[f'{col}_change'] = df_sample.groupby('customer_ID')[col].pct_change()
        df_sample[f'{col}_trend'] = df_sample.groupby('customer_ID')[col].diff()
        df_sample[f'{col}_3m_avg'] = df_sample.groupby('customer_ID')[col].transform(
            lambda x: x.rolling(3, min_periods=1).mean()
        )
        feature_count += 3

# Balance trends
for col in balance_cols[:3]:
    if col in df_sample.columns:
        df_sample[f'{col}_change'] = df_sample.groupby('customer_ID')[col].pct_change()
        df_sample[f'{col}_growth'] = df_sample.groupby('customer_ID')[col].diff()
        df_sample[f'{col}_volatility'] = df_sample.groupby('customer_ID')[col].transform(
            lambda x: x.rolling(3, min_periods=1).std()
        )
        feature_count += 3

print(f"✓ Created {feature_count} time series features")

In [ ]:
# Aggregate to customer level
print("\nAggregating to customer level...")

# Select columns to aggregate
agg_dict = {}

# Original features
for col in numeric_cols[:20]:  # Limit to prevent memory issues
    if col != 'target' and col in df_sample.columns:
        agg_dict[col] = ['mean', 'std', 'min', 'max']

# Trend features
trend_cols = [col for col in df_sample.columns if '_change' in col or '_trend' in col or '_growth' in col]
for col in trend_cols:
    agg_dict[col] = ['mean', 'std']

# Target
if 'target' in df_sample.columns:
    agg_dict['target'] = 'first'

# Aggregate
df_agg = df_sample.groupby('customer_ID').agg(agg_dict).reset_index()

# Flatten column names
df_agg.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                  for col in df_agg.columns.values]

# Fill NaN
df_agg = df_agg.fillna(0)

print(f"✓ Aggregated dataset shape: {df_agg.shape}")
print(f"✓ Total features for modeling: {df_agg.shape[1] - 2}")

---
<a id='section5'></a>
## 5. Model Training

Training three ML models for insurance risk prediction.

In [ ]:
# Prepare features and target
target_col = 'target_first' if 'target_first' in df_agg.columns else 'target'

if target_col not in df_agg.columns:
    raise ValueError("No target column found! Check data loading.")

X = df_agg.drop(['customer_ID', target_col], axis=1, errors='ignore')
y = df_agg[target_col]

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nDefault rate: {y.mean():.1%}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTraining models...")

In [ ]:
# Model 1: Logistic Regression
print("\n[1/3] Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_pred_proba = lr_model.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_pred_proba)
print(f"  AUC: {lr_auc:.4f}")

In [ ]:
# Model 2: Random Forest
print("\n[2/3] Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, 
    class_weight='balanced', n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred_proba)
print(f"  AUC: {rf_auc:.4f}")

In [ ]:
# Model 3: Gradient Boosting
print("\n[3/3] Gradient Boosting...")
gb_model = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, random_state=42
)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_pred_proba = gb_model.predict_proba(X_test)[:, 1]
gb_auc = roc_auc_score(y_test, gb_pred_proba)
print(f"  AUC: {gb_auc:.4f}")

In [ ]:
# Model comparison
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'AUC Score': [lr_auc, rf_auc, gb_auc]
})

print("\n" + "="*60)
print("MODEL COMPARISON RESULTS")
print("="*60)
print(results.to_string(index=False))

# Select best model
best_idx = results['AUC Score'].idxmax()
best_model_name = results.loc[best_idx, 'Model']
best_pred_proba = [lr_pred_proba, rf_pred_proba, gb_pred_proba][best_idx]
best_pred = [lr_pred, rf_pred, gb_pred][best_idx]

print(f"\n✓ Best performing model: {best_model_name}")

In [ ]:
# Detailed metrics for best model
print(f"\n{best_model_name} - Classification Report:")
print("="*60)
print(classification_report(y_test, best_pred, target_names=['No Default', 'Default']))

---
<a id='section6'></a>
## 6. Insurance Metrics Calculation

Calculating key insurance metrics: premiums, expected loss, loss ratios.

In [ ]:
# Insurance parameters
AVG_CLAIM_AMOUNT = 5000  # Average credit card balance at default
PROFIT_MARGIN = 0.15      # 15% profit margin
EXPENSE_RATIO = 0.25      # 25% for administrative expenses

print("Insurance Product Parameters:")
print(f"  Average claim amount: ${AVG_CLAIM_AMOUNT:,}")
print(f"  Profit margin: {PROFIT_MARGIN:.0%}")
print(f"  Expense ratio: {EXPENSE_RATIO:.0%}")

In [ ]:
# Calculate expected loss and premiums
expected_loss = best_pred_proba * AVG_CLAIM_AMOUNT
premium = expected_loss * (1 + PROFIT_MARGIN + EXPENSE_RATIO)

# Aggregate metrics
total_claims = expected_loss.sum()
total_premiums = premium.sum()
loss_ratio = (total_claims / total_premiums) * 100
profit = total_premiums - total_claims

print("\n" + "="*60)
print("INSURANCE PRODUCT ANALYSIS")
print("="*60)
print(f"\nPer Customer Metrics:")
print(f"  Average expected loss: ${expected_loss.mean():,.2f}")
print(f"  Average premium: ${premium.mean():,.2f}")

print(f"\nAggregate Metrics:")
print(f"  Total expected claims: ${total_claims:,.2f}")
print(f"  Total premium revenue: ${total_premiums:,.2f}")
print(f"  Expected profit: ${profit:,.2f}")

print(f"\nKey Ratios:")
print(f"  Loss ratio: {loss_ratio:.1f}%")
print(f"  Profit margin: {(profit/total_premiums)*100:.1f}%")

In [ ]:
# Risk segmentation
risk_df = pd.DataFrame({
    'Risk_Score': best_pred_proba,
    'Premium': premium,
    'Expected_Loss': expected_loss,
    'Actual_Default': y_test.values
})

risk_df['Risk_Category'] = pd.cut(
    risk_df['Risk_Score'],
    bins=[0, 0.2, 0.4, 0.6, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Summary by risk category
risk_summary = risk_df.groupby('Risk_Category').agg({
    'Premium': 'mean',
    'Expected_Loss': 'mean',
    'Actual_Default': ['count', 'sum', 'mean']
}).round(2)

risk_summary.columns = ['Avg Premium ($)', 'Avg Expected Loss ($)', 
                        'N Customers', 'Actual Defaults', 'Default Rate']

print("\n" + "="*60)
print("RISK-BASED PREMIUM STRUCTURE")
print("="*60)
print(risk_summary)

---
<a id='section7'></a>
## 7. Early Intervention Analysis

Analyzing the cost-benefit of proactive customer intervention.

In [ ]:
# Intervention parameters
INTERVENTION_THRESHOLD = 0.4   # Risk score above which to intervene
INTERVENTION_COST = 200        # Cost per intervention (counseling, etc.)
INTERVENTION_SUCCESS_RATE = 0.30  # 30% of interventions prevent default

print("Early Intervention Program Parameters:")
print(f"  Intervention threshold: {INTERVENTION_THRESHOLD:.0%} predicted risk")
print(f"  Cost per intervention: ${INTERVENTION_COST}")
print(f"  Success rate: {INTERVENTION_SUCCESS_RATE:.0%}")

In [ ]:
# Identify high-risk customers
high_risk_mask = best_pred_proba >= INTERVENTION_THRESHOLD
n_high_risk = high_risk_mask.sum()
pct_high_risk = (n_high_risk / len(best_pred_proba)) * 100

print("\n" + "="*60)
print("INTERVENTION OPPORTUNITY ANALYSIS")
print("="*60)
print(f"\nHigh-Risk Customer Identification:")
print(f"  Customers above threshold: {n_high_risk}")
print(f"  Percentage of portfolio: {pct_high_risk:.1f}%")

# Among high-risk customers, how many actually defaulted?
high_risk_defaults = ((best_pred_proba >= INTERVENTION_THRESHOLD) & (y_test == 1)).sum()
print(f"  Actual defaults in group: {high_risk_defaults}")
print(f"  Potential intervention opportunity: {high_risk_defaults} customers")

In [ ]:
# Calculate intervention economics
prevented_defaults = high_risk_defaults * INTERVENTION_SUCCESS_RATE
intervention_costs = n_high_risk * INTERVENTION_COST
claims_saved = prevented_defaults * AVG_CLAIM_AMOUNT
net_savings = claims_saved - intervention_costs
roi = (net_savings / intervention_costs) * 100 if intervention_costs > 0 else 0

print(f"\nIntervention Economics:")
print(f"  Prevented defaults (est.): {prevented_defaults:.0f}")
print(f"  Total intervention cost: ${intervention_costs:,.2f}")
print(f"  Claims savings: ${claims_saved:,.2f}")
print(f"  Net savings: ${net_savings:,.2f}")
print(f"  ROI: {roi:.0f}%")

print(f"\n💡 Key Insight: For every $1 spent on intervention, save ${claims_saved/intervention_costs:.2f} in claims!")

---
<a id='section8'></a>
## 8. Visualizations

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))

# 1. Model Performance Comparison
ax1 = plt.subplot(3, 3, 1)
models = ['Logistic\nRegression', 'Random\nForest', 'Gradient\nBoosting']
aucs = [lr_auc, rf_auc, gb_auc]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax1.bar(models, aucs, color=colors)
ax1.set_ylabel('AUC Score', fontsize=11, fontweight='bold')
ax1.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax1.set_ylim([0, 1])
for bar, auc in zip(bars, aucs):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{auc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 2. Feature Importance (Random Forest)
ax2 = plt.subplot(3, 3, 2)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-10:]
ax2.barh(range(len(indices)), importances[indices], color='#2ecc71')
ax2.set_yticks(range(len(indices)))
ax2.set_yticklabels([X.columns[i] for i in indices], fontsize=8)
ax2.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax2.set_title('Top 10 Feature Importance', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# 3. Confusion Matrix
ax3 = plt.subplot(3, 3, 3)
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax3, cbar=False, 
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
ax3.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax3.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax3.set_title(f'Confusion Matrix ({best_model_name})', fontsize=13, fontweight='bold')

# 4. Risk Score Distribution
ax4 = plt.subplot(3, 3, 4)
ax4.hist(best_pred_proba[y_test==0], bins=30, alpha=0.6, label='No Default', color='#3498db')
ax4.hist(best_pred_proba[y_test==1], bins=30, alpha=0.6, label='Default', color='#e74c3c')
ax4.axvline(INTERVENTION_THRESHOLD, color='red', linestyle='--', linewidth=2, label='Intervention Threshold')
ax4.set_xlabel('Predicted Default Probability', fontsize=11, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax4.set_title('Risk Score Distribution', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

# 5. Premium by Risk Category
ax5 = plt.subplot(3, 3, 5)
risk_avg_premium = risk_df.groupby('Risk_Category')['Premium'].mean()
colors_risk = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
bars = ax5.bar(risk_avg_premium.index, risk_avg_premium.values, color=colors_risk)
ax5.set_ylabel('Average Premium ($)', fontsize=11, fontweight='bold')
ax5.set_title('Premium Structure by Risk', fontsize=13, fontweight='bold')
ax5.set_xlabel('Risk Category', fontsize=11, fontweight='bold')
for bar in bars:
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height + 50,
             f'${height:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax5.grid(axis='y', alpha=0.3)

# 6. Customer Distribution
ax6 = plt.subplot(3, 3, 6)
risk_counts = risk_df['Risk_Category'].value_counts()
wedges, texts, autotexts = ax6.pie(risk_counts.values, labels=risk_counts.index, 
                                     autopct='%1.1f%%', colors=colors_risk, startangle=90)
ax6.set_title('Customer Distribution by Risk', fontsize=13, fontweight='bold')
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

# 7. Financial Projection
ax7 = plt.subplot(3, 3, 7)
categories = ['Premium\nRevenue', 'Expected\nClaims', 'Expected\nProfit']
values = [total_premiums, total_claims, profit]
colors_fin = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax7.bar(categories, values, color=colors_fin)
ax7.set_ylabel('Amount ($)', fontsize=11, fontweight='bold')
ax7.set_title('Insurance Product Financials', fontsize=13, fontweight='bold')
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax7.text(bar.get_x() + bar.get_width()/2., height + 500,
             f'${value:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax7.grid(axis='y', alpha=0.3)

# 8. Model Calibration
ax8 = plt.subplot(3, 3, 8)
decile_df = pd.DataFrame({'y_test': y_test, 'pred_proba': best_pred_proba})
decile_df['decile'] = pd.qcut(decile_df['pred_proba'], 10, labels=False, duplicates='drop')
decile_default_rate = decile_df.groupby('decile')['y_test'].mean() * 100
ax8.plot(decile_default_rate.index, decile_default_rate.values, 
         marker='o', linewidth=2, markersize=8, color='#e74c3c')
ax8.set_xlabel('Risk Decile (0=Low, 9=High)', fontsize=11, fontweight='bold')
ax8.set_ylabel('Actual Default Rate (%)', fontsize=11, fontweight='bold')
ax8.set_title('Model Calibration', fontsize=13, fontweight='bold')
ax8.grid(alpha=0.3)

# 9. Intervention Impact
ax9 = plt.subplot(3, 3, 9)
scenarios = ['Current\nDefaults', 'Prevented\nDefaults', 'Net\nDefaults']
counts = [high_risk_defaults, prevented_defaults, high_risk_defaults - prevented_defaults]
colors_int = ['#e74c3c', '#2ecc71', '#f39c12']
bars = ax9.bar(scenarios, counts, color=colors_int)
ax9.set_ylabel('Number of Defaults', fontsize=11, fontweight='bold')
ax9.set_title('Early Intervention Impact', fontsize=13, fontweight='bold')
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax9.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{int(count)}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax9.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('insurance_analysis_results.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved: insurance_analysis_results.png")
plt.show()

---
<a id='section9'></a>
## 9. Key Findings & Conclusions

In [ ]:
print("="*70)
print("KEY FINDINGS SUMMARY")
print("="*70)

print("\n1. MODEL PERFORMANCE:")
print(f"   • Best model: {best_model_name}")
print(f"   • AUC Score: {max(lr_auc, rf_auc, gb_auc):.3f}")
print(f"   • Demonstrates strong predictive capability for default risk")

print("\n2. INSURANCE PRODUCT VIABILITY:")
print(f"   • Average premium: ${premium.mean():,.2f} per customer")
print(f"   • Loss ratio: {loss_ratio:.1f}% (target: <80%)")
print(f"   • Expected profit margin: {(profit/total_premiums)*100:.1f}%")
print(f"   • Risk-based pricing ranges from ${risk_avg_premium.min():.0f} to ${risk_avg_premium.max():.0f}")

print("\n3. EARLY INTERVENTION OPPORTUNITY:")
print(f"   • {pct_high_risk:.1f}% of customers flagged for intervention")
print(f"   • Potential to prevent {prevented_defaults:.0f} defaults")
print(f"   • Net savings: ${net_savings:,.2f}")
print(f"   • ROI: {roi:.0f}% on intervention program")

print("\n4. BEHAVIORAL PATTERNS:")
print(f"   • Time series features capture payment deterioration")
print(f"   • Risk scores effectively separate defaulters from non-defaulters")
print(f"   • Model calibration shows strong predictive accuracy across risk deciles")

print("\n5. INSURANCE PRODUCT RECOMMENDATIONS:")
print(f"   ✓ Implement risk-based pricing (vs. flat-rate)")
print(f"   ✓ Launch early intervention program at {INTERVENTION_THRESHOLD:.0%} risk threshold")
print(f"   ✓ Combine insurance coverage with financial counseling")
print(f"   ✓ Monthly premium adjustments based on behavioral trends")
print(f"   ✓ Target high-risk customers with proactive support")

print("\n" + "="*70)
print("CONCLUSION")
print("="*70)
print("\nBehavioral monitoring and machine learning enable insurance providers to:")
print("  1. More accurately price payment protection insurance")
print("  2. Identify at-risk customers before default occurs")
print("  3. Reduce insurance losses through timely intervention")
print("  4. Improve customer outcomes through proactive support")
print("\nThis approach represents a shift from reactive claims processing to")
print("predictive risk management, benefiting both insurers and policyholders.")
print("="*70)

---

## Next Steps for Paper Writing

Use the results from this notebook to write your term paper:

1. **Copy key numbers** from the outputs above
2. **Reference the visualization** (insurance_analysis_results.png)
3. **Follow the paper outline** provided separately
4. **Focus on insurance implications** of each finding
5. **Submit this notebook** along with your paper

**Remember:** The code demonstrates methodology. The paper should emphasize insurance applications and business insights!

---

## Novel Technique Note

This notebook uses **time series feature engineering** for credit risk prediction - creating temporal trend features (payment trajectories, utilization growth rates, volatility measures) rather than treating each month independently. This behavioral monitoring approach is the novel technique for this project.